# Project 29: Unsupervised Software Anomaly Detection

**Team No.:** 22  
**Team Members:** Akash Mahalik; Piyash Acharya; Diptiranjan Mahakud; SoumyaRanjan Das  
**Proposed Hybrid Model:** TCN Autoencoder + Transformer Autoencoder + Deep SVDD  
**Dataset:** System log corpora — https://www.kaggle.com/datasets/krishd123/log-data-for-anomaly-detection

Self-contained Google Colab workflow. Run cells top to bottom; outputs are generated from the downloaded data and are intentionally not pre-populated.

## 0. Setup — Environment, Imports, Reproducibility

In [ ]:
!pip -q install kagglehub transformers sentencepiece torch-geometric captum pennylane tqdm tabulate
import os, json, random, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda:0")
print('Device:', DEVICE)


### CONFIG

In [ ]:
CONFIG={
 'project_no':'29','project_name':'Unsupervised Software Anomaly Detection','team_no':'22','team_members':'Akash Mahalik; Piyash Acharya; Diptiranjan Mahakud; SoumyaRanjan Das',
 'task_type':'anomaly_detection','dataset_name':'System log corpora','kaggle_dataset_slug':'krishd123/log-data-for-anomaly-detection',
 'proposed_model':'TCN Autoencoder + Transformer Autoencoder + Deep SVDD','target_column':None,'id_columns':[],'time_column':None,
 'split_ratios':{'train':.70,'val':.15,'test':.15},'random_seed':SEED,
 'data_raw_dir':'data/29/raw','data_processed_dir':'data/29/processed','figures_dir':'data/29/figures','results_dir':'data/29/results','reports_dir':'data/29/reports',
 'batch_size':64,'epochs':20,'patience':4,'learning_rate':1e-3,'max_rows':120000
}
for key in ['data_raw_dir','data_processed_dir','figures_dir','results_dir','reports_dir']: Path(CONFIG[key]).mkdir(parents=True,exist_ok=True)
CONFIG


## 1. Dataset Download

In [ ]:
import kagglehub, shutil
cache_path=Path(kagglehub.dataset_download(CONFIG['kaggle_dataset_slug']))
raw_dir=Path(CONFIG['data_raw_dir'])
for src in cache_path.rglob('*'):
    if src.is_file():
        dst=raw_dir/src.relative_to(cache_path); dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists(): shutil.copy2(src,dst)
raw_files=[p for p in raw_dir.rglob('*') if p.is_file()]
assert raw_files, 'Dataset download produced no files.'
assert sum(p.stat().st_size for p in raw_files)>1024, 'Downloaded payload is unexpectedly small.'
print(f'Discovered {len(raw_files)} files; total bytes={sum(p.stat().st_size for p in raw_files):,}')


## 2. Load Raw Data

In [ ]:
files=[p for p in Path(CONFIG['data_raw_dir']).rglob('*') if p.is_file() and p.stat().st_size>0]; assert files,'No log files found.'
rows=[]
for path in sorted(files,key=lambda p:p.stat().st_size,reverse=True)[:12]:
 try:
  if path.suffix.lower()=='.csv':
   part=pd.read_csv(path,nrows=CONFIG['max_rows']//12); text_col=next((c for c in part if any(k in c.lower() for k in ['content','message','log','event'])),part.columns[0]); label_col=next((c for c in part if any(k==c.lower() for k in ['label','anomaly'])),None)
   for i,t in enumerate(part[text_col].astype(str)): rows.append((path.name,i,t,None if label_col is None else part.iloc[i][label_col]))
  else:
   with open(path,errors='ignore') as f:
    for i,line in enumerate(f):
     if i>=CONFIG['max_rows']//12: break
     rows.append((path.name,i,line.strip(),None))
 except Exception as e: print('Skipped',path.name,type(e).__name__)
df=pd.DataFrame(rows,columns=['source','position','message','provided_label']); assert len(df)>=100,'Insufficient parseable log events.'; df=df[df.message.str.len()>0].drop_duplicates(subset=['source','position']).reset_index(drop=True); target='provided_label'; CONFIG['target_column']=target; print(df.shape,df.source.value_counts().head())


## 3. Exploratory Data Analysis (EDA) + Data Quality Memo

In [ ]:
print('Shape:',df.shape); print('Sources:',df.source.nunique()); print('Duplicates:',df.duplicated().sum()); memo=f"""# Data Quality Memo

- {len(df):,} events from {df.source.nunique()} source files.
- Unsupervised training uses only the earliest 70% of each source.
- Labels, where present, are reserved strictly for final evaluation.
"""; Path('reports/data_quality_memo.md').write_text(memo,encoding='utf-8')


## 4. Preprocessing & Feature Engineering

The following split cell creates the partitions first and fits all learned preprocessing artifacts on the training partition only.

## 5. Train / Validation / Test Split

In [ ]:
df=df.sort_values(['source','position']); label_text=df.provided_label.astype(str).str.strip().str.lower(); df['is_anomaly']=np.where(df.provided_label.notna(),label_text.isin(['1','anomaly','abnormal','true','error','attack']).astype(int),np.nan); train_parts=[]; val_parts=[]; test_parts=[]
for _,g in df.groupby('source'):
 a=int(.7*len(g)); b=int(.85*len(g)); train_parts.append(g.iloc[:a]); val_parts.append(g.iloc[a:b]); test_parts.append(g.iloc[b:])
train_df=pd.concat(train_parts); val_df=pd.concat(val_parts); test_df=pd.concat(test_parts); assert train_df.groupby('source').position.max().lt(test_df.groupby('source').position.min()).all()
if train_df.is_anomaly.notna().any(): train_df=train_df[train_df.is_anomaly==0].copy()
from sklearn.feature_extraction.text import HashingVectorizer
vectorizer=HashingVectorizer(n_features=256,alternate_sign=False,norm=None,analyzer='char',ngram_range=(3,5)); scaler=StandardScaler().fit(vectorizer.transform(train_df.message).toarray())
def make_windows(frame,length=12):
 xs=[]; labels=[]
 for _,g in frame.groupby('source',sort=False):
  values=scaler.transform(vectorizer.transform(g.message).toarray()).astype('float32'); labs=g.is_anomaly.to_numpy()
  for end in range(length,len(values)+1): xs.append(values[end-length:end]); labels.append(labs[end-1])
 return np.stack(xs),np.asarray(labels,dtype='float32')
X_train,L_train=make_windows(train_df); X_val,L_val=make_windows(val_df); X_test,L_test=make_windows(test_df); feature_names=[f'char_hash_{i}' for i in range(X_train.shape[2])]; Path('data/processed/split_manifest.json').write_text(json.dumps({'strategy':'per-source chronological windows','window_length':X_train.shape[1],'train_windows':len(X_train),'val_windows':len(X_val),'test_windows':len(X_test),'normal_only_training_when_labels_available':True},indent=2))


## 6. PyTorch Dataset & DataLoader

In [ ]:
class LogDataset(Dataset):
 def __init__(self,x): self.x=torch.tensor(x)
 def __len__(self): return len(self.x)
 def __getitem__(self,i): return (self.x[i],)
train_loader=DataLoader(LogDataset(X_train),batch_size=CONFIG['batch_size'],shuffle=True); val_loader=DataLoader(LogDataset(X_val),batch_size=CONFIG['batch_size']); test_loader=DataLoader(LogDataset(X_test),batch_size=CONFIG['batch_size'])


## 7. Model Definitions

In [ ]:
class TCNTransformerDeepSVDD(nn.Module):

    def __init__(self, d, h=64):
        super().__init__()
        self.tcn = nn.Sequential(nn.Conv1d(d, h, 3, padding=2, dilation=2), nn.ReLU(), nn.Conv1d(h, h, 3, padding=4, dilation=4), nn.ReLU())
        layer = nn.TransformerEncoderLayer(h, 4, 128, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, 2)
        self.latent = nn.Linear(h, 16)
        self.decoder = nn.Linear(16, d)
        self.register_buffer('center', torch.zeros(16))

    def forward(self, x):
        t = self.tcn(x.transpose(1, 2))[..., :x.shape[1]].transpose(1, 2)
        h = self.transformer(t).mean(1)
        z = self.latent(h)
        return (self.decoder(z), z)
hybrid = TCNTransformerDeepSVDD(X_train.shape[2])

def reconstruction_loss(model, b):
    recon, z = model(b[0])
    return nn.functional.mse_loss(recon, b[0].mean(1))

def svdd_loss(model, b):
    recon, z = model(b[0])
    return 0.25 * nn.functional.mse_loss(recon, b[0].mean(1)) + ((z - model.center) ** 2).mean()


## 8. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, loss_fn, checkpoint, epochs=None):
    epochs = epochs or CONFIG['epochs']
    model = model.to(DEVICE)
    grouped = hasattr(model, 'parameter_groups')
    opt = torch.optim.AdamW(model.parameter_groups() if grouped else model.parameters(), lr=CONFIG['learning_rate'], weight_decay=0.0001)
    total_steps = max(1, epochs * len(train_loader))
    warmup_steps = max(1, int(0.1 * total_steps))
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lambda step: min((step + 1) / warmup_steps, 1.0)) if grouped else torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=2, factor=0.5)
    best = float('inf')
    stale = 0
    history = {'train_loss': [], 'val_loss': []}
    for epoch in tqdm(range(epochs), desc='Training', unit='epoch'):
        model.train()
        total = 0
        for batch in train_loader:
            batch = [v.to(DEVICE) for v in batch]
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model, batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if grouped:
                scheduler.step()
            total += loss.item() * batch[0].shape[0]
        train_loss = total / len(train_loader.dataset)
        model.eval()
        total = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = [v.to(DEVICE) for v in batch]
                total += loss_fn(model, batch).item() * batch[0].shape[0]
        val_loss = total / len(val_loader.dataset)
        if not grouped:
            scheduler.step(val_loss)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        print(f'Epoch {epoch + 1:02d}: train={train_loss:.4f}, val={val_loss:.4f}')
        if val_loss < best - 1e-05:
            best = val_loss
            stale = 0
            torch.save(model.state_dict(), checkpoint)
        else:
            stale += 1
            if stale >= CONFIG['patience']:
                break
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
    return history
pretrain_history = train_model(hybrid, train_loader, val_loader, reconstruction_loss, 'results/pretrained_hybrid.pt', max(3, CONFIG['epochs'] // 2))
hybrid = hybrid.to(DEVICE)
with torch.no_grad():
    centers = []
    for x, in train_loader:
        centers.append(hybrid(x.to(DEVICE))[1].cpu())
    hybrid.center.copy_(torch.cat(centers).mean(0).to(DEVICE))
hybrid_history = train_model(hybrid, train_loader, val_loader, svdd_loss, 'results/best_hybrid.pt')


## 9. Evaluation Metrics

In [ ]:
def scores(model, loader):
    model.eval()
    out = []
    with torch.no_grad():
        for x, in loader:
            x = x.to(DEVICE)
            recon, z = model(x)
            s = ((recon - x.mean(1)) ** 2).mean(1)
            if hasattr(model, 'center'):
                s = s + 0.1 * ((z - model.center) ** 2).mean(1)
            out.extend(s.cpu().numpy())
    return np.array(out)
hval = scores(hybrid, val_loader)
hybrid_scores = scores(hybrid, test_loader)
hthr = float(np.quantile(hval, 0.99))
hybrid_pred = (hybrid_scores > hthr).astype(int)
known = ~np.isnan(L_test)
results = {'hybrid': {'threshold': hthr, 'flag_rate': float(hybrid_pred.mean()), 'median_score': float(np.median(hybrid_scores))}}
if known.any():
    y = L_test[known].astype(int)
    for name, s, pred in [('hybrid', hybrid_scores[known], hybrid_pred[known])]:
        if np.unique(y).size == 2:
            results[name].update({'roc_auc': float(roc_auc_score(y, s)), 'average_precision': float(average_precision_score(y, s)), 'f1': float(precision_recall_fscore_support(y, pred, average='binary', zero_division=0)[2])})
        else:
            results[name]['label_metric_note'] = 'Held-out labels contain one class; ROC-AUC, AP, and F1 are undefined.'
Path('results/metrics.json').write_text(json.dumps(results, indent=2))
print(results)


## 10. Required Figures

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hybrid_history['val_loss'], label='Hybrid')
plt.legend()
plt.tight_layout()
plt.savefig('figures/fig01_loss_curves.png', dpi=150)
plt.show()
plt.figure(figsize=(8, 4))
sns.histplot(hybrid_scores, bins=60)
plt.axvline(hthr, color='r', label='threshold')
plt.legend()
plt.tight_layout()
plt.savefig('figures/fig02_anomaly_scores.png', dpi=150)
plt.show()
plt.figure(figsize=(8, 4))
plt.plot(hybrid_scores)
plt.axhline(hthr, color='r')
plt.xlabel('Chronological test event')
plt.tight_layout()
plt.savefig('figures/fig03_score_timeline.png', dpi=150)
plt.show()
sample = torch.tensor(X_test[:128], device=DEVICE, requires_grad=True)
recon, z = hybrid(sample)
score = ((recon - sample.mean(1)) ** 2).mean(1).sum()
score.backward()
importance = (sample.grad * sample).abs().mean((0, 1)).detach().cpu().numpy()
top = np.argsort(importance)[-20:]
plt.figure(figsize=(8, 5))
plt.barh(np.array(feature_names)[top], importance[top])
plt.tight_layout()
plt.savefig('figures/fig04_feature_importance.png', dpi=150)
plt.show()
worst = np.argsort(hybrid_scores)[-20:]
plt.figure(figsize=(8, 4))
plt.bar(range(len(worst)), hybrid_scores[worst])
plt.ylabel('Anomaly score')
plt.tight_layout()
plt.savefig('figures/fig05_error_analysis.png', dpi=150)
plt.show()
comparison_name = 'average_precision' if known.any() and np.unique(L_test[known]).size == 2 else 'median_score'
plt.figure(figsize=(7, 4))
plt.bar(['hybrid'], [results['hybrid'][comparison_name]])
plt.ylabel(comparison_name)
plt.title('Proposed model held-out performance')
plt.tight_layout()
plt.savefig('figures/fig06_proposed_metrics.png', dpi=150)
plt.show()
